In [0]:
# %sql
# DROP TABLE IF EXISTS workspace.futbol.silver_players;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.futbol.silver_players (
    match_id STRING COMMENT 'ID único del partido extraído de la URL',
    team_name STRING COMMENT 'Nombre del equipo',
    player_name STRING COMMENT 'Nombre del jugador',
    player_position STRING COMMENT 'Posición del jugador'
)
USING DELTA
COMMENT 'Tabla Silver de los jugadores de los partidos de fútbol'
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'quality' = 'silver'
)

In [0]:
%sql
MERGE INTO workspace.futbol.silver_players AS target
USING (
    SELECT
        match_id,
        team_name,
        player_name,
        player_position
    FROM (
        SELECT
            REPLACE(substring_index(match[0].url, '-', -1), '/', '') AS match_id,
            team.name AS team_name,
            team_players.name AS player_name,
            team_players.roleName AS player_position,
            ROW_NUMBER() OVER (PARTITION BY REPLACE(substring_index(match[0].url, '-', -1), '/', ''), team.name, team_players.name ORDER BY team_players.roleName) AS rn
        FROM
            workspace.futbol.bronze_matchs
            LATERAL VIEW EXPLODE(match[1].`@graph`) t AS team
            LATERAL VIEW EXPLODE(team.athlete) p AS team_players
        WHERE
            team.`@type` = 'SportsTeam'
    )
    WHERE rn = 1
) AS source
ON target.match_id = source.match_id
   AND target.team_name = source.team_name
   AND target.player_name = source.player_name
WHEN MATCHED THEN
    UPDATE SET
        target.player_position = source.player_position
WHEN NOT MATCHED THEN
    INSERT (
        match_id,
        team_name,
        player_name,
        player_position
    )
    VALUES (
        source.match_id,
        source.team_name,
        source.player_name,
        source.player_position
    )